In [8]:
from collections import Counter
from pathlib import Path

folder = Path("dataset/normal")

extensions = Counter(
    p.suffix.lower() or "<no extension>"
    for p in folder.iterdir()
    if p.is_file()
)

extensions

Counter({'.wav': 1648, '.mp3': 40})

In [ ]:
import os
import torch
import librosa
import numpy as np
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim

class Spectrogram(Dataset):
    def __init__(self, base_path, sr=16000, window_size=5.0, min_overlap=1.5): #sr=sample rate
        self.sr = sr
        self.window_size = int(window_size * sr) #we convert time into array indices
        self.min_overlap = int(min_overlap * sr)
        self.samples = []

        for label, category in enumerate(['normal', 'malicious']):
            dir_path = os.path.join(base_path, category)
            for f in os.listdir(dir_path):
                if f.lower().endswith(('.wav', '.mp3')):
                    path = os.path.join(dir_path, f)
                    duration_sec = librosa.get_duration(path=path)
                    audio = int(sr*duration_sec)
                    print(f"seconds: {duration_sec}; audio: {audio}\n")

                    # Logic: Calculate step based on 8.5s rule
                    if audio < int(8.5 * sr) and audio >= self.window_size:
                        step = (audio - self.window_size) // 1
                        if step==0: step=1
                    else:
                        step = self.window_size - self.min_overlap

                    # Sliding Window Loop
                    for start in range(0, audio - self.window_size + 1, step):
                        self.samples.append((path, start, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, start, label = self.samples[idx]
        audio, _ = librosa.load(path, sr=self.sr, offset=start/self.sr, duration=5.0)

        # Spectrogram conversion
        mel_spec = librosa.feature.melspectrogram(y=audio, sr=self.sr, n_mels=128)
        log_mel = librosa.power_to_db(mel_spec, ref=np.max)

        return torch.tensor(log_mel).unsqueeze(0), torch.tensor(label)
p=Spectrogram('dataset')

ModuleNotFoundError: No module named 'torch'

In [ ]:

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        # Block 1: Feature Extraction
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )

        # Block 2: The "Bulletproof" Funnel
        # This forces the spatial dimensions down to 1x1, regardless of input length.
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # Block 3: Classification
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, 128), # It's 32 because the last Conv2d output 32 channels
            nn.ReLU(),
            nn.Dropout(0.5), # Added dropout to prevent overfitting
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x



In [ ]:
# Initialization and Splitting
dataset = Spectrogram('dataset')
train_sz = int(0.8 * len(dataset))
val_sz = int(0.1 * len(dataset))
test_sz = len(dataset) - train_sz - val_sz

train_db, val_db, test_db = random_split(dataset, [train_sz, val_sz, test_sz])

train_loader = DataLoader(train_db, batch_size=16, shuffle=True)
model = SimpleCNN()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
if device is "cpu" :
  print("ERROR! Not using GPU")
  exit(0)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


epochs = 10 # How many times to loop over the whole dataset

print(f"Starting training on {device}...")

for epoch in range(epochs):
    model.train() # Put model in training mode
    running_loss = 0.0

    for batch_idx, (spectrograms, labels) in enumerate(train_loader):
        # Move data to the same device as the model
        spectrograms = spectrograms.to(device)
        labels = labels.to(device)

        # 1. Zero the gradients (clear old memory)
        optimizer.zero_grad()

        # 2. Forward pass (make a prediction)
        outputs = model(spectrograms)

        # 3. Calculate the error
        loss = criterion(outputs, labels)

        # 4. Backward pass (calculate how to fix the error)
        loss.backward()

        # 5. Update the weights
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader):.4f}")

# NOW you save it, after it has learned
torch.save(model.state_dict(), 'malicious_call_detector_trained.pth')
print("Training complete and model saved.")

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd drive/MyDrive/MaliciousCallDetection/


/content/drive/MyDrive/MaliciousCallDetection


In [15]:
!git commit -m 'Modified MinIO_Process to work in google colab folder hierarchy. Also 16kHz data is in drive now'

[main a3c1f38] Modified MinIO_Process to work in google colab folder hierarchy. Also 16kHz data is in drive now
 2 files changed, 2 insertions(+), 2 deletions(-)


In [14]:
!git config --global user.email "marinescudragos2014@gmail.com"
!git config --global user.name "Marinescu Dragos"

In [16]:
!git push

fatal: could not read Username for 'https://github.com': No such device or address
